In [66]:
import requests
import importlib
from bs4 import BeautifulSoup
import pandas as pd
import snowflake.connector
import sqlalchemy as db
import numpy as np
import import_ipynb
import re
from datetime import datetime, timedelta
import snowflake_functions as sf  # Import the notebook as a module
import transform_data as td 
import extract_data as ed 
from skills_array import get_skills_array
skills_array = get_skills_array() # get full list of skills

In [83]:
# upload latest function
importlib.reload(sf)
importlib.reload(ed)
importlib.reload(td)

# connection snowflake
engine = sf.connect_snowflake()
connection = engine.connect()

# intialise tables that will later be loaded to snowflake
job_table = pd.DataFrame(columns=['JOB_ID', 'TITLE','COMPANY','LOCATION','EMPLOYMENT_TYPE','SALARY','PAY_PERIOD','POST_DATE'])
job_skills_table = pd.DataFrame(columns=["skill_id", "skill_name"])


# get the job_id of the last entry in the jobs table
job_id = sf.last_job_id(connection) + 1

# go through many pages
max_pages = 1
for j in range(max_pages):
    
    # get all job URL links
    full_urls = ed.get_URLs_to_jobs(j) 
    
    # go through all the URLs and extract relevant data.
    for k, url in enumerate(full_urls):   
        
        # Scrape job URL.
        response = requests.get(url) 
        soup = BeautifulSoup(response.content, "html.parser") 
        
        # Enter job ad in to jobs table data frame
        job_props, req_skills = ed.get_job_data(soup, skills_array) # extract the information from job
        parsed_salary = td.parse_salary(job_props[4]) # clean up salary   
        job_table = td.update_job_table(job_props, parsed_salary, job_table, job_id + k) # put into data frame.


        # get that entries id, get skill ids and enter it into job_skills table.
        df_skill_ids = sf.get_skill_ids(req_skills, connection)
        job_skills_table = td.update_job_skills_table(job_props, parsed_salary, job_skills_table, df_skill_ids, job_id + k)
        

# upload tables to snowflake database
job_table.to_sql('jobs', con=engine, if_exists='append', index=False)
job_skills_table.to_sql('job_skills', con=engine, if_exists='append', index=False)

# Close connection
connection.close()

Connected to Snowflake!
https://www.seek.com.au/job/82335381?type=standard&ref=search-standalone
https://www.seek.com.au/job/82340497?type=standard&ref=search-standalone
https://www.seek.com.au/job/82397329?type=standard&ref=search-standalone
https://www.seek.com.au/job/82305280?type=standard&ref=search-standalone
https://www.seek.com.au/job/82257256?type=standard&ref=search-standalone
https://www.seek.com.au/job/82253004?type=standard&ref=search-standalone
https://www.seek.com.au/job/82379770?type=standard&ref=search-standalone
https://www.seek.com.au/job/82379683?type=standard&ref=search-standalone
https://www.seek.com.au/job/82294554?type=standard&ref=search-standalone
https://www.seek.com.au/job/82399006?type=standard&ref=search-standalone
https://www.seek.com.au/job/82320209?type=standard&ref=search-standalone
https://www.seek.com.au/job/82105910?type=standard&ref=search-standalone
https://www.seek.com.au/job/82321927?type=standard&ref=search-standalone
https://www.seek.com.au/job

In [68]:
# Close connection
connection.close()

In [77]:
importlib.reload(sf)
importlib.reload(ed)
importlib.reload(td)

engine = sf.connect_snowflake()
connection = engine.connect()
job_skills_table.to_sql('job_skills', con=engine, if_exists='append', index=False)

# Close connection
connection.close()

Connected to Snowflake!


In [82]:
job_table

,JOB_ID,TITLE,COMPANY,LOCATION,EMPLOYMENT_TYPE,SALARY,PAY_PERIOD,POST_DATE
0,1523,Data Engineer,Talent Insights Group Pty Ltd,Sydney NSW,Contract/Temp,None,None,2025-02-25
1,1524,Data Engineer | Mid-Level,Billigence,Sydney NSW,Contract/Temp,None,None,2025-02-25
2,1525,Junior Data Engineer/Analyst,RASSURE,"Chatswood, Sydney NSW",Full time,None,None,None
3,1526,Senior Data Engineer,Talenza,Sydney NSW,Full time,160000,annually,2025-02-25
4,1527,Data Engineer,Cuscal Limited (SR),Sydney NSW,Full time,None,None,2025-02-21
5,1528,Senior Data Engineer (Contexa),FinXL IT Professional Services,Sydney NSW,Contract/Temp,None,None,None
6,1529,Data Engineering Manager,Talenza,Sydney NSW,Full time,200000,annually,None
7,1530,Machine Learning & Data Engineer,Fundo Loans,Sydney NSW,Full time,140000,annually,2025-02-24
8,1531,Senior Data Engineer - AI Initiatives,FourQuarters Recruitment,Sydney NSW,Full time,200000,annually,2025-02-19
9,1532,Lead Data Engineer / Scientist - Up to $220K +...,Private Advertiser,Sydney NSW,Full time,220000,annually,None
